In [2]:
set.seed(102191) # Semilla para reproducibilidad

N_SIMULACIONES <- 1000
TIROS <- 100

# Distribución base de habilidades del pelotón (p_i ~ Unif(0.50, 0.75))
generar_p_reales <- function(n) {
  return(runif(n, min = 0.50, max = 0.75))
}

# --- 1. Algoritmo de eliminación iterativa para C1 y C2 ---
simular_eliminacion <- function(num_jugadoras_iniciales) {
  p_reales <- generar_p_reales(num_jugadoras_iniciales)

  # Proceso de eliminación paso a paso (elimina 10 por ronda)
  while (length(p_reales) > 10) {
    # Todas las jugadoras sobrevivientes tiran 100 veces
    aciertos <- rbinom(length(p_reales), size = TIROS, prob = p_reales)

    # Ordenar de menor a mayor acierto
    orden <- order(aciertos)

    # Eliminar las 10 peores de la ronda
    p_reales <- p_reales[orden[11:length(p_reales)]]
  }

  # Ronda final con las últimas 10 jugadoras
  aciertos_finales <- rbinom(10, size = TIROS, prob = p_reales)

  # Registramos el promedio de aciertos de las 10 finalistas y su rendimiento real esperado
  mejor_idx <- which.max(aciertos_finales)
  return(p_reales[mejor_idx])
}

# --- 2. Algoritmo para C4 (50 rondas de 100 tiros para 2 jugadoras) ---
simular_c4 <- function() {
  p_reales <- generar_p_reales(2)

  # 50 rondas de 100 tiros cada una
  rondas_j1 <- rbinom(50, size = TIROS, prob = p_reales[1])
  rondas_j2 <- rbinom(50, size = TIROS, prob = p_reales[2])

  prom_j1 <- mean(rondas_j1)
  prom_j2 <- mean(rondas_j2)

  # Se selecciona la que obtuvo el mayor promedio acumulado
  mejor_idx <- if (prom_j1 >= prom_j2) 1 else 2
  return(p_reales[mejor_idx])
}

# --- Ejecución del Monte Carlo ---
p_c1 <- numeric(N_SIMULACIONES)
p_c2 <- numeric(N_SIMULACIONES)
p_c4 <- numeric(N_SIMULACIONES)

for (i in 1:N_SIMULACIONES) {
  p_c1[i] <- simular_eliminacion(num_jugadoras_iniciales = 100)
  p_c2[i] <- simular_eliminacion(num_jugadoras_iniciales = 200)
  p_c4[i] <- simular_c4()
}

# --- Parámetros fijos ---
p_c5 <- 0.696  # 69.6% explícito
p_c6 <- 0.790  # 79.0% insesgado
p_c7 <- 0.790  # 79.0% insesgado

# --- Consolidación de Resultados ---
resultados <- data.frame(
  Candidata = c(
    "C2 (Mejor de 200 con eliminación)",
    "C1 (Mejor de 100 con eliminación)",
    "C5 (Sobrina - 1000 tiros)",
    "C4 (Mejor de 2 en 50 rondas)",
    "C7 (Reemplazo directo 79/100)",
    "C6 (Quiniela 79/100)"
  ),
  Habilidad_Real_Promedio = c(
    mean(p_c2),
    mean(p_c1),
    p_c5,
    mean(p_c4),
    p_c7,
    p_c6
  ),
  Encestes_Esperados_Nueva_Ronda = c(
    mean(p_c2) * 100,
    mean(p_c1) * 100,
    p_c5 * 100,
    mean(p_c4) * 100,
    p_c7 * 100,
    p_c6 * 100
  )
)

# Ordenar de menor a mayor rendimiento esperado
resultados <- resultados[order(resultados$Habilidad_Real_Promedio), ]

print(resultados, row.names = FALSE)

                         Candidata Habilidad_Real_Promedio
      C4 (Mejor de 2 en 50 rondas)               0.6640349
         C5 (Sobrina - 1000 tiros)               0.6960000
 C1 (Mejor de 100 con eliminación)               0.7341812
 C2 (Mejor de 200 con eliminación)               0.7382247
     C7 (Reemplazo directo 79/100)               0.7900000
              C6 (Quiniela 79/100)               0.7900000
 Encestes_Esperados_Nueva_Ronda
                       66.40349
                       69.60000
                       73.41812
                       73.82247
                       79.00000
                       79.00000
